# 41 - Locked held-out NBIS test: L1 vs triplet

This notebook implements `41_triplet_heldout_test_protocol.md`.
It does not train, recalibrate thresholds or evaluate FVC2004.

1. Use the same Colab GPU/runtime as notebook 40. If reset, run 00 first
   for pinned dependencies and the official archive. Notebook 00 does not open test images.
2. `RUN_HELDOUT = True` is set in the first code cell for this requested run.
   Run all only when ready. Do not edit the analysis after viewing test results.
3. If interrupted before completion, rerun with the same configuration to resume.
   A completed run is not rerun. Use the saved reports under
   `evaluations/native_triplet_heldout_test_v1`.

The primary decision uses only `dry_like/severe`: macro FRR must decrease and
FAR must not increase in any of the six databases. Clean, mild and moderate
are secondary. Test images and templates stay in the Colab runtime; Drive
receives scores, extraction metadata and aggregate reports, not image/template archives.


In [ ]:
RUN_HELDOUT = True
if not RUN_HELDOUT:
    raise RuntimeError('Held-out test is still sealed. Read the protocol, then explicitly enable this run.')


In [ ]:
if globals().get('RUN_HELDOUT') is not True:
    raise RuntimeError('Run the explicit held-out confirmation cell first.')
import ast
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import re
import shutil
import subprocess
import sys
import types
import zipfile
import io
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT, BOZORTH3 = (NBIS_ROOT / 'install' / 'bin' / name for name in ('mindtct', 'bozorth3'))
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel')
CONDITIONS = [('identity', 'none')] + [('dry_like', level) for level in ('mild', 'moderate', 'severe')]
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'alternative_corruption_validation_v1' / '476fc096ee2d51f03921'
REPLICATION_ROOT = PROJECT_ROOT / 'experiments' / 'm3_l1_initialization_v1'
PRIOR_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / '1374b21b3b10b3202a8d'
COMPARISONS = [('input', 'm0'), ('input', 'l1_control'), ('input', 'l1_sobel'), ('l1_control', 'l1_sobel')]

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact checksum mismatch: {directory / name}')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Frozen held-out NBIS test; no training.')
RISK_ROOT = PROJECT_ROOT / 'experiments' / 'counterfactual_risk_pilot_v1'
PRIOR24_DIR = PROJECT_ROOT / 'evaluations' / 'dry_specialization_validation_v1' / 'a2bcd5551cf4a97e3bf2'

import time
import platform
from pathlib import PurePosixPath
PILOT_RUNS=('native_l1','native_triplet')
METHODS=PILOT_RUNS
PILOT_ROOT=PROJECT_ROOT/'experiments'/'native_triplet_replication_v1'
COMPARISONS=[('native_l1','native_triplet')]

PROTOCOL_SHA = '7949a624b49e0aa39806c468e489596cd97f35f49be19e225f71579b901e96f4'
VALIDATION_SHA = 'b1ddff0a86f722ea06634f784050b6d8cfb6a0e37a0a91f010acc499df6e4127'
TRAIN_SHA = '59ad0c66eee00fe6e050cb02749083cc39ebb7af7a31a599a450c94d4e4a7d16'
VALIDATION_DIR = PROJECT_ROOT/'evaluations'/'native_triplet_replication_validation_v1'/VALIDATION_SHA[:20]
REPORT_DIR = PROJECT_ROOT/'evaluations'/'native_triplet_heldout_test_v1'
DATA_ROOT = LOCAL_ROOT/'heldout_41_data'
PAIR_COUNTS = {
    ('FVC2000','DB1'):29, ('FVC2000','DB2'):17, ('FVC2000','DB3'):29,
    ('FVC2002','DB1'):20, ('FVC2002','DB2'):27, ('FVC2002','DB3'):23,
}
THRESHOLDS = {
    ('FVC2000','DB1'):(22,22), ('FVC2000','DB2'):(19,20), ('FVC2000','DB3'):(16,17),
    ('FVC2002','DB1'):(23,24), ('FVC2002','DB2'):(20,22), ('FVC2002','DB3'):(20,30),
}
def require_incomplete_run():
    if globals().get('RUN_HELDOUT') is not True:
        raise RuntimeError('Held-out test requires explicit confirmation.')
    if (REPORT_DIR/'completion.json').exists():
        raise RuntimeError(f'Held-out run already completed. Read the saved reports: {REPORT_DIR}')

require_incomplete_run()


In [ ]:
WORK_DIR=LOCAL_ROOT/'native_pilot_nbis'
REPORT_DIR=WORK_DIR/'reports'
WORK_DIR.mkdir(parents=True,exist_ok=True)
REPORT_DIR.mkdir(parents=True,exist_ok=True)
NBIS_URL = 'https://nigos.nist.gov/nist/nbis/nbis_v5_0_0.zip'
NBIS_SHA256 = '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
NBIS_BYTES = 52595795
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
NBIS_ARCHIVE = ARCHIVE_DIR / 'nbis_v5_0_0.zip'
missing_tools = [name for name in ('curl', 'gcc', 'make', 'sh') if shutil.which(name) is None]
if missing_tools:
    raise RuntimeError(f'Missing build tools: {missing_tools}. Install required tools before retrying.')
if not NBIS_ARCHIVE.exists():
    partial = LOCAL_ROOT / 'nbis_v5_0_0.zip.part'
    if not partial.exists() or partial.stat().st_size != NBIS_BYTES:
        subprocess.run([
            'curl', '--fail', '--location', '--continue-at', '-',
            '--connect-timeout', '30', '--max-time', '600',
            '--output', str(partial), NBIS_URL,
        ], check=True)
    if partial.stat().st_size != NBIS_BYTES or file_digest(partial) != NBIS_SHA256:
        raise RuntimeError('NBIS download integrity failure. Partial file retained; do not build it.')
    pending = NBIS_ARCHIVE.with_suffix('.zip.part')
    shutil.copyfile(partial, pending)
    if file_digest(pending) != NBIS_SHA256:
        raise RuntimeError('NBIS Drive copy checksum mismatch.')
    pending.replace(NBIS_ARCHIVE)
    partial.unlink()
if NBIS_ARCHIVE.stat().st_size != NBIS_BYTES or file_digest(NBIS_ARCHIVE) != NBIS_SHA256:
    raise RuntimeError('Cached NBIS archive differs from the verified release.')
print('Official NBIS source verified:', NBIS_SHA256)
BUILD_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
SOURCE_ROOT = BUILD_ROOT / 'source'
PREFIX = BUILD_ROOT / 'install'
SOURCE = SOURCE_ROOT / 'Rel_5.0.0'
MINDTCT = PREFIX / 'bin' / 'mindtct'
BOZORTH3 = PREFIX / 'bin' / 'bozorth3'
STAMP = BUILD_ROOT / 'verified_build.json'
build_commands = [
    ['sh', 'setup.sh', str(PREFIX), '--without-X11', '--without-OPENJP2', '--64'],
    ['make', 'config'],
    ['make', 'it', 'CC=gcc -fcommon'],
    ['make', 'install', 'LIBNBIS=no', 'CC=gcc -fcommon'],
]
build_signature = {'source_sha256': NBIS_SHA256, 'commands': build_commands}
if STAMP.exists():
    build_record = json.loads(STAMP.read_text(encoding='utf-8'))
    if build_record['signature'] != build_signature:
        raise RuntimeError('Cached NBIS build configuration differs; do not reuse it.')
    for binary in (MINDTCT, BOZORTH3):
        if not binary.is_file() or file_digest(binary) != build_record['binary_sha256'][binary.name]:
            raise RuntimeError(f'Cached NBIS binary changed or is missing: {binary}')
    print('Reusing verified local NBIS build.')
else:
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    PREFIX.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NBIS_ARCHIVE) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename:
                raise RuntimeError(f'Unsafe source archive path: {member.filename}')
        archive.extractall(SOURCE_ROOT)
    for path in SOURCE.rglob('*'):
        if path.is_file() and (path.name == 'configure' or path.suffix == '.sh'):
            path.chmod(path.stat().st_mode | 0o111)
    build_log = WORK_DIR / 'build.log'
    with build_log.open('w', encoding='utf-8') as output:
        for command in build_commands:
            print('Building:', ' '.join(command), flush=True)
            result = subprocess.run(command, cwd=SOURCE, stdout=output, stderr=subprocess.STDOUT)
            output.flush()
            if result.returncode:
                shutil.copyfile(build_log, REPORT_DIR / 'build.log')
                print('\n'.join(build_log.read_text(errors='replace').splitlines()[-60:]))
                raise RuntimeError(f'NBIS build failed with code {result.returncode}. See saved build.log.')
    shutil.copyfile(build_log, REPORT_DIR / 'build.log')
    build_record = {
        'signature': build_signature,
        'binary_sha256': {binary.name: file_digest(binary) for binary in (MINDTCT, BOZORTH3)},
        'compiler': subprocess.run(['gcc', '--version'], check=True, capture_output=True, text=True).stdout,
        'platform': platform.platform(),
    }
    STAMP.write_text(json.dumps(build_record, indent=2), encoding='utf-8')
versions = {}
for binary in (MINDTCT, BOZORTH3):
    result = subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)
    versions[binary.name] = result.stdout + result.stderr
    print(binary.name, versions[binary.name].strip())
(REPORT_DIR / 'build_provenance.json').write_text(
    json.dumps({**build_record, 'versions': versions}, indent=2), encoding='utf-8')

In [ ]:

def checked_json(path, expected):
    if file_digest(path) != expected:
        raise RuntimeError(f'Frozen configuration checksum differs: {path}')
    return json.loads(path.read_text(encoding='utf-8'))

validation_config = checked_json(VALIDATION_DIR/'configuration.json', VALIDATION_SHA)
PILOT_DIR = PILOT_ROOT/TRAIN_SHA[:20]
pilot_config = checked_json(PILOT_DIR/'configuration.json', TRAIN_SHA)
pilot_completion = checked_json(PILOT_DIR/'completion.json', validation_config['pilot_completion_sha256'])
if (validation_config['pilot_configuration_sha256'] != TRAIN_SHA
        or validation_config['methods'] != list(METHODS)
        or validation_config['conditions'] != [list(c) for c in CONDITIONS]
        or pilot_config['seed'] != 20261127 or pilot_config['run_names'] != list(METHODS)
        or pilot_completion['configuration_sha256'] != TRAIN_SHA
        or pilot_completion['status'] != 'native_counterfactual_pilot_complete'):
    raise RuntimeError('Expected the frozen Notebook 40 and Notebook 39 run.')
verify_files(PILOT_DIR, pilot_completion['file_sha256'])
split_protocol = checked_json(SPLIT_DIR/'protocol.json', validation_config['split_protocol_sha256'])
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
pair_config = checked_json(PAIR_DIR/'configuration.json', validation_config['pair_configuration_sha256'])
sim_config = checked_json(SIM_DIR/'configuration.json', validation_config['simulator_configuration_sha256'])
if (file_digest(SIM_DIR/'fingerprint_degradation.py') != sim_config['module_sha256']
        or build_record['binary_sha256'] != validation_config['nbis_binary_sha256']
        or build_record['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['mindtct_flags'] != [] or pair_config['bozorth3_flags'] != ['-A','minminutiae=10']
        or any(c['split_protocol_sha256'] != validation_config['split_protocol_sha256']
               for c in (pair_config, sim_config))):
    raise RuntimeError('Frozen simulator, split or NBIS provenance differs.')
for key, value in [('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
                   ('numpy_version', np.__version__), ('pillow_version', Image.__version__)]:
    if validation_config[key] != value or pilot_config[key] != value:
        raise RuntimeError(f'Frozen dependency differs: {key}. Restore it with Notebook 00.')
m0_config = json.loads((M0_DIR/'configuration.json').read_text(encoding='utf-8'))
verify_files(M0_DIR, m0_config['source_file_sha256'])
training = load_module('heldout41_model', M0_DIR/'restoration_baseline.py')
simulator = load_module('heldout41_simulator', SIM_DIR/'fingerprint_degradation.py')
helper_path = VALIDATION_DIR/'frozen_helpers.py.txt'
if file_digest(helper_path) != validation_config['frozen_helpers_sha256']:
    raise RuntimeError('Frozen NBIS helper checksum differs.')
HELPER_SOURCE = helper_path.read_text(encoding='utf-8')


In [ ]:
def load_selected(directory,pointer,expected_sha,run):
    name=pointer['file']
    if (Path(name).name!=name or '/' in name or '\\' in name or name in ('.','..')
            or pointer['configuration_sha256']!=expected_sha
            or file_digest(directory/name)!=pointer['sha256']):
        raise RuntimeError('Selected model pointer/integrity differs.')
    state=torch.load(directory/name,map_location='cpu',weights_only=True)
    if (state['configuration_sha256']!=expected_sha or state['run']!=run
            or state['completed_epochs']!=pointer['completed_epochs']):
        raise RuntimeError('Selected model payload/epoch differs.')
    with torch.random.fork_rng(devices=[]):
        model=training.RestorationUNet()
    model.load_state_dict(state['model'],strict=True)
    if not all(torch.isfinite(value).all() for value in model.state_dict().values()):
        raise RuntimeError('Nonfinite selected weights.')
    return model.to(DEVICE).eval()

model_pointers = validation_config['model_checkpoints']
if set(model_pointers) != set(METHODS):
    raise RuntimeError('Expected exactly two locked model pointers.')
models = {}
for name, epoch in zip(METHODS, (1, 2)):
    directory = PILOT_DIR/name
    pointer = model_pointers[name]
    marker = checked_json(directory/'completion.json', pilot_completion['run_completion_sha256'][name])
    if (marker['configuration_sha256'] != TRAIN_SHA or marker['run'] != name
            or marker['epochs_completed'] != 2 or marker['best_checkpoint'] != pointer
            or pointer['file'] != 'selected.pt' or pointer['completed_epochs'] != epoch):
        raise RuntimeError(f'Frozen checkpoint selection differs: {name}')
    models[name] = load_selected(directory, pointer, TRAIN_SHA, name)


In [ ]:
def make_pairs(frame):
    rows = []
    for (dataset, db), group in frame.groupby(['dataset', 'db'], sort=True):
        group = group.sort_values(['identity_key', 'impression_id'])
        if group.duplicated(['identity_key', 'impression_id']).any():
            raise RuntimeError('Duplicate identity/impression keys in validation.')
        for identity, finger in group.groupby('identity_key'):
            if set(finger['impression_id']) != set(range(1, 9)):
                raise RuntimeError(f'Expected impressions 1-8: {identity}')
        gallery = group[group['impression_id'].eq(1)].to_dict('records')
        probes = group[~group['impression_id'].eq(1)].to_dict('records')
        distinct_paths = set(group.drop_duplicates(['identity_key', 'sha256'])['relative_path'])
        if len(gallery) < 2:
            raise RuntimeError(f'Need at least two validation fingers: {dataset} {db}')
        for probe in probes:
            for enrollment in gallery:
                same = probe['identity_key'] == enrollment['identity_key']
                pair_key = probe['relative_path'] + '|' + enrollment['relative_path']
                rows.append({
                    'pair_id': hashlib.sha256(pair_key.encode('utf-8')).hexdigest(),
                    'dataset': dataset, 'db': db, 'label': int(same),
                    'probe_path': probe['relative_path'], 'gallery_path': enrollment['relative_path'],
                    'probe_identity': probe['identity_key'], 'gallery_identity': enrollment['identity_key'],
                    'duplicate_sensitivity_eligible': (
                        probe['relative_path'] in distinct_paths and enrollment['relative_path'] in distinct_paths),
                })
    result = pd.DataFrame(rows)
    if result['pair_id'].duplicated().any():
        raise RuntimeError('Duplicate pair IDs.')
    return result

locked = pd.read_csv(SPLIT_DIR/'manifest_splits.csv', keep_default_na=False)
heldout = locked[locked['split'].eq('test_synthetic')].sort_values('relative_path').reset_index(drop=True)
expected = {key: 8*count for key, count in PAIR_COUNTS.items()}
if (heldout.groupby(['dataset','db']).size().to_dict() != expected
        or heldout['relative_path'].duplicated().any()
        or set(heldout['relative_path']) & set(locked.loc[~locked['split'].eq('test_synthetic'), 'relative_path'])
        or set(heldout['leakage_group']) & set(locked.loc[~locked['split'].eq('test_synthetic'), 'leakage_group'])):
    raise RuntimeError('Unexpected held-out membership, count or split leakage.')
for row in heldout.itertuples():
    if (not re.fullmatch(rf'{row.dataset}/{row.db}_[AB]/[0-9]+_[0-9]+\.(?:tif|bmp)', row.relative_path, re.IGNORECASE)
            or not re.fullmatch(r'[0-9a-f]{64}', row.sha256)):
        raise RuntimeError('Unsafe or unexpected held-out image metadata.')
pairs = make_pairs(heldout)
probe_records = heldout[heldout['impression_id'].ne(1)].to_dict('records')
gallery_records = heldout[heldout['impression_id'].eq(1)].to_dict('records')
expected_pairs = {(dataset, db, label): 7*n*(1 if label else n-1)
                  for (dataset, db), n in PAIR_COUNTS.items() for label in (0, 1)}
if (pairs.groupby(['dataset','db','label']).size().to_dict() != expected_pairs
        or (len(gallery_records), len(probe_records), len(pairs)) != (145, 1015, 25403)):
    raise RuntimeError('Frozen held-out pair counts differ.')


In [ ]:

require_incomplete_run()
evaluation_config = dict(version='native_triplet_heldout_test_v1', protocol_sha256=PROTOCOL_SHA,
    implementation_sha256='fb4fcdea88a28c35c2b433828105a3afc1d481deea0ef52896d73774b251a9d5', split='test_synthetic',
    validation_configuration_sha256=VALIDATION_SHA, pilot_configuration_sha256=TRAIN_SHA,
    model_checkpoints=model_pointers, methods=list(METHODS), conditions=CONDITIONS,
    pair_csv_sha256=hashlib.sha256(pairs.to_csv(index=False, lineterminator='\n').encode()).hexdigest(),
    thresholds=[dict(dataset=d, db=b, native_l1=v[0], native_triplet=v[1])
                for (d,b),v in sorted(THRESHOLDS.items())])
evaluation_sha = hashlib.sha256(json_bytes(evaluation_config)).hexdigest()
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json':json_bytes(evaluation_config), 'pairs.csv':
                   pairs.to_csv(index=False, lineterminator='\n').encode()}.items():
    path = REPORT_DIR/name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Cannot resume a different held-out protocol: {path}')
    if not path.exists():
        write_verified(path, data)
CACHE_DIR = LOCAL_ROOT/'heldout_41_nbis'/evaluation_sha[:20]
TEMPLATE_DIR = CACHE_DIR/'templates'
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
active_helpers=types.ModuleType('native_nb41_helpers')
active_helpers.__dict__.update(globals())
exec(compile(HELPER_SOURCE,'<frozen-nbis-helpers>','exec'),active_helpers.__dict__)
# Keep the frozen loader's validation, but do not persist image/template archives.
helper_functions = {node.name: node for node in ast.parse(HELPER_SOURCE).body
                    if isinstance(node, ast.FunctionDef)}
loader = ast.get_source_segment(HELPER_SOURCE, helper_functions['load_unit'])
archive_entry = ", 'templates_and_logs.zip'"
if loader.count(archive_entry) != 1:
    raise RuntimeError('Expected the original frozen NBIS completion schema.')
exec(compile(loader.replace(archive_entry, ''), '<heldout-unit-loader>', 'exec'), active_helpers.__dict__)
units38={}
heldout_paths=frozenset(heldout.relative_path)
def condition_input41(record,family,severity):
    if record['relative_path'] not in heldout_paths or (family,severity) not in CONDITIONS:
        raise RuntimeError('Evaluation may open only frozen held-out images/conditions.')
    dpi=pair_config['source_dpi'][f"{record['dataset']}_{record['db']}"]
    reference=training.normalized_image(str(DATA_ROOT/record['relative_path']),dpi)
    if family=='identity':
        return reference,reference.copy()
    degraded,_=simulator.degrade(reference,family,severity,
        simulator.case_seed(record['relative_path'],family,severity,sim_config['seed']))
    return reference,degraded
active_helpers.condition_input=condition_input41


In [ ]:
def safe_relative(relative):
    if not isinstance(relative,str) or not relative or '\\' in relative or ':' in relative:
        raise RuntimeError(f'Unsafe frozen relative path: {relative!r}')
    path=PurePosixPath(relative)
    if not path.parts or path.is_absolute() or '..' in path.parts or path.as_posix()!=relative:
        raise RuntimeError(f'Unsafe frozen relative path: {relative!r}')
    return path

def safe_destination(root,relative):
    destination=root/Path(*safe_relative(relative).parts)
    if root.resolve() not in destination.resolve().parents:
        raise RuntimeError(f'Runtime path escapes its declared root: {relative}')
    return destination

def restore_test_images(records,data_root,archive_supplier):
    needed=[]
    for row in records:
        path=safe_destination(data_root,row['relative_path'])
        if not path.is_file() or file_digest(path)!=row['sha256']:
            needed.append(row)
    if not needed:
        print(f'All{len(records)} runtime held-out files already verified; no archive access.')
        return dict(required=len(records),restored=0,reused=len(records))
    print(f'Restoring/repairing {len(needed)} runtime copies; frozen Drive artifacts stay untouched.',flush=True)
    archive_path=archive_supplier()
    pattern=re.compile(r'^74034_3_En_4_MOESM1_ESM/(FVC2000|FVC2002|FVC2004)/Dbs/'
        r'DB([123])_([AB])/([0-9]+_[0-9]+\.(?:tif|bmp))$',re.IGNORECASE)
    with zipfile.ZipFile(archive_path) as archive:
        mapping={}
        for member in archive.infolist():
            match=pattern.fullmatch(member.filename)
            if not match:
                continue
            year,db,subset,filename=match.groups()
            relative=f'{year.upper()}/DB{db}_{subset.upper()}/{filename}'
            if ((member.external_attr>>16)&0o170000) not in (0,0o100000):
                raise RuntimeError(f'Unsafe non-regular/symlink archive member: {relative}')
            if relative in mapping:
                raise RuntimeError(f'Duplicate canonical archive member: {relative}')
            mapping[relative]=member
        required_bytes=sum(mapping[row['relative_path']].file_size for row in needed if row['relative_path'] in mapping)
        if shutil.disk_usage(data_root).free<required_bytes+64*1024*1024:
            raise RuntimeError('Not enough ephemeral disk space to restore selected images.')
        for index,row in enumerate(needed,1):
            relative=row['relative_path']
            if relative not in mapping:
                raise RuntimeError(f'Frozen image missing from official archive: {relative}')
            member=mapping[relative]
            if member.file_size>32*1024*1024:
                raise RuntimeError(f'Unexpected fingerprint member size: {relative}')
            destination=safe_destination(data_root,relative)
            destination.parent.mkdir(parents=True,exist_ok=True)
            pending=destination.with_suffix(destination.suffix+'.prepare.part')
            with archive.open(member) as src,pending.open('wb') as dst:
                shutil.copyfileobj(src,dst)
            if file_digest(pending)!=row['sha256']:
                raise RuntimeError(f'Frozen image checksum mismatch: {relative}; no replacement published.')
            pending.replace(destination)
            if index%256==0 or index==len(needed):
                print(f'Verified restored images {index}/{len(needed)}',flush=True)
    return dict(required=len(records),restored=len(needed),reused=len(records)-len(needed))

def official_archive():
    candidates = (LOCAL_ROOT/'prepare_after_reset'/'fvc_official.zip',
                  PROJECT_ROOT/'archives'/'fvc2000_2002_2004_official.zip')
    for path in candidates:
        if path.is_file() and path.stat().st_size == 869928818:
            digest = hashlib.md5()
            with path.open('rb') as stream:
                for block in iter(lambda: stream.read(8*1024*1024), b''):
                    digest.update(block)
            if digest.hexdigest() == '0412227e3a8c99ab9b0022b8aef9912d':
                return path
            print(f'Official archive checksum mismatch: {path}')
    raise RuntimeError('No verified official archive. Run Notebook 00 and retry.')

require_incomplete_run()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(restore_test_images(heldout.to_dict('records'), DATA_ROOT, official_archive))


In [ ]:
require_incomplete_run()

pending=[(family,severity,method) for family,severity in CONDITIONS for method in METHODS
    if active_helpers.load_unit(REPORT_DIR/f'{family}_{severity}_{method}') is None]
gallery_frame=None
if pending:
    gallery_rows=[]
    for index,record in enumerate(gallery_records,1):
        reference,_=condition_input41(record,'identity','none')
        gallery_rows.append({**record,**active_helpers.extract_png(active_helpers.quantize(reference))})
        if index%50==0 or index==len(gallery_records):
            print(f'Original galleries {index}/{len(gallery_records)}',flush=True)
    gallery_frame=pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR/'gallery_extraction.csv',gallery_frame)
for family,severity in CONDITIONS:
    for method in METHODS:
        unit=f'{family}_{severity}_{method}'
        directory=REPORT_DIR/unit
        loaded=active_helpers.load_unit(directory)
        if loaded is not None:
            units38[unit]=loaded
            print('Verified/resumed NBIS variant:',unit,flush=True)
            continue
        directory.mkdir(parents=True,exist_ok=True)
        work=CACHE_DIR/unit
        work.mkdir(parents=True,exist_ok=True)
        active_helpers.model=models[method]
        rows=[]
        started38=time.monotonic()
        for index,record in enumerate(probe_records,1):
            reference,degraded=condition_input41(record,family,severity)
            output=active_helpers.restore_image(degraded)
            rows.append({**record,**active_helpers.extract_png(active_helpers.quantize(output)),
                **active_helpers.pixel_metrics(reference,degraded,output),
                'condition':f'{family}/{severity}','method':method})
            if index%100==0 or index==len(probe_records):
                print(f'{unit}: images {index}/{len(probe_records)}, {(time.monotonic()-started38)/60:.1f}min',flush=True)
        extraction=pd.DataFrame(rows)
        scored=active_helpers.match_pairs(extraction,gallery_frame,work)
        write_csv(directory/'scores.csv',scored)
        write_csv(directory/'extraction.csv',extraction)
        marker=dict(configuration_sha256=evaluation_sha,variant=unit,
            file_sha256={name:file_digest(directory/name) for name in
                ('scores.csv','extraction.csv')})
        write_verified(directory/'completion.json',json_bytes(marker))
        units38[unit]=active_helpers.load_unit(directory)
        if units38[unit] is None:
            raise RuntimeError(f'Variant did not persist: {unit}')
        print('Saved NBIS variant:',unit,flush=True)


In [ ]:
def paired_attempt_changes(before,after,thresholds,condition,baseline,method):
    columns=['pair_id','dataset','db','label','matchable','score']
    joined=before[columns].merge(after[columns],on='pair_id',suffixes=('_before','_after'),
        how='outer',validate='one_to_one',indicator=True)
    if not joined['_merge'].eq('both').all() or any(
            not joined[f'{key}_before'].equals(joined[f'{key}_after']) for key in ('dataset','db','label')):
        raise RuntimeError('Paired NBIS metadata/attempts differ.')
    rows=[]
    for (dataset,db),group in joined.groupby(['dataset_before','db_before'],sort=True):
        threshold_before,threshold_after=thresholds[(dataset,db)]
        accepted_before=group.matchable_before & group.score_before.ge(threshold_before)
        accepted_after=group.matchable_after & group.score_after.ge(threshold_after)
        genuine=group.label_before.eq(1)
        impostor=~genuine
        rows.append(dict(condition=condition,baseline=baseline,method=method,dataset=dataset,db=db,
            false_rejects_l1=int((genuine & ~accepted_before).sum()),
            false_rejects_triplet=int((genuine & ~accepted_after).sum()),
            false_accepts_l1=int((impostor & accepted_before).sum()),
            false_accepts_triplet=int((impostor & accepted_after).sum()),
            genuine_attempts=int(genuine.sum()),impostor_attempts=int(impostor.sum()),
            genuine_recoveries=int((genuine & ~accepted_before & accepted_after).sum()),
            genuine_harms=int((genuine & accepted_before & ~accepted_after).sum()),
            newly_accepted_impostors=int((impostor & ~accepted_before & accepted_after).sum()),
            newly_rejected_impostors=int((impostor & accepted_before & ~accepted_after).sum())))
    return rows


In [ ]:
from fractions import Fraction


def primary_decision(rows):
    severe = [row for row in rows if row['condition'] == 'dry_like/severe']
    if (len(severe) != len(PAIR_COUNTS)
            or {(row['dataset'], row['db']) for row in severe} != set(PAIR_COUNTS)):
        raise RuntimeError('Cannot decide without all six primary DB rows.')
    frr_delta = sum(Fraction(row['false_rejects_triplet']-row['false_rejects_l1'],
                             row['genuine_attempts']) for row in severe) / len(severe)
    far_increased = [f"{row['dataset']}/{row['db']}" for row in severe
                     if row['false_accepts_triplet'] > row['false_accepts_l1']]
    return dict(passed=frr_delta < 0 and not far_increased,
                macro_delta_frr_pp=float(frr_delta*100), far_increased_databases=far_increased)


require_incomplete_run()
if len(units38) != len(CONDITIONS)*len(METHODS):
    raise RuntimeError('Incomplete held-out variants.')
metric_rows, paired_rows, pixel_rows = [], [], []
for family, severity in CONDITIONS:
    condition = f'{family}/{severity}'
    for index, method in enumerate(METHODS):
        scores, extraction = units38[f'{family}_{severity}_{method}']
        if scores.groupby(['dataset','db','label']).size().to_dict() != expected_pairs:
            raise RuntimeError('Scored pair counts differ from the frozen pair list.')
        for (dataset, db), frame in scores.groupby(['dataset','db'], sort=True):
            threshold = THRESHOLDS[(dataset,db)][index]
            for view, selected in [('all_impressions', frame),
                    ('byte_distinct_sensitivity', frame[frame.duplicate_sensitivity_eligible])]:
                values = active_helpers.metrics_at_threshold(selected, threshold)
                metric_rows.append(dict(condition=condition, method=method, dataset=dataset,
                                        db=db, view=view, **values))
        for (dataset, db), frame in extraction.groupby(['dataset','db'], sort=True):
            pixel_rows.append(dict(condition=condition, method=method, dataset=dataset, db=db,
                                   images=len(frame), unmatchable_images=int(frame.status.ne('ok').sum())))
    baseline, method = METHODS
    paired_rows.extend(paired_attempt_changes(
        units38[f'{family}_{severity}_{baseline}'][0], units38[f'{family}_{severity}_{method}'][0],
        THRESHOLDS, condition, baseline, method))
metrics = pd.DataFrame(metric_rows)
macro = metrics.groupby(['condition','method','view'], sort=True).agg(
    databases=('db','size'), attempt_frr=('attempt_frr','mean'), attempt_far=('attempt_far','mean')).reset_index()
if len(macro) != 16 or not macro.databases.eq(6).all() or len(paired_rows) != 24:
    raise RuntimeError('Incomplete held-out report coverage.')
decision = primary_decision(paired_rows)
for name, frame in [('test_metrics.csv',metrics), ('macro_metrics.csv',macro),
                    ('paired_attempt_changes.csv',pd.DataFrame(paired_rows)),
                    ('image_failures.csv',pd.DataFrame(pixel_rows))]:
    write_csv(REPORT_DIR/name,frame)


In [ ]:

require_incomplete_run()
for name, pointer in model_pointers.items():
    if file_digest(PILOT_DIR/name/pointer['file']) != pointer['sha256']:
        raise RuntimeError(f'Frozen checkpoint changed during run: {name}')
report_names = ('configuration.json','pairs.csv','gallery_extraction.csv','test_metrics.csv',
                'macro_metrics.csv','paired_attempt_changes.csv','image_failures.csv')
summary = dict(status='heldout_test_complete', configuration_sha256=evaluation_sha,
               protocol_sha256=PROTOCOL_SHA, external_test_evaluated=False,
               variants=len(units38), primary=decision,
               unit_completion_sha256={key:file_digest(REPORT_DIR/key/'completion.json') for key in units38},
               file_sha256={name:file_digest(REPORT_DIR/name) for name in report_names})
write_verified(REPORT_DIR/'completion.json', json_bytes(summary))
print(macro[macro['view'].eq('all_impressions')].to_string(index=False))
print('SEVERE:', 'PASS' if decision['passed'] else 'FAIL', decision)
print('NBIS41 HELD-OUT TEST COMPLETE:', REPORT_DIR)
